# Solving the Krusell-Smith (1998) Model in Julia

This notebook solves the Krusell-Smith (1998) model using value function iteration (VFI). The notebook is structured as follows:
1. Model setup (household, firm, equilibrium approximation).
2. Numerical algorithm description.
3. Julia implementation (VFI + simulation + law-of-motion update).

## 1) Original Model and Equilibrium

### HH problem

Individual state is $s=(k,e,z,\Phi)$, where $k$ is assets, $e\in\{0,1\}$ is employment, $z\in\{z_b,z_g\}$ is aggregate productivity, and $\Phi$ is the distribution of wealth and employment across agents.

Given prices $r(z,\Phi)$ and $w(z,\Phi)$, and law of motion for the distribution $\Phi'$, the Bellman equation is
$$
V(k,e,z,\Phi)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',\Phi')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,\Phi))k + w(z,\Phi)e \\
k'\ge k_{min} \\
\Phi' = \mathcal{H}(\Phi,z,z')
$$

However, the distribution $\Phi$ is infinite-dimensional, so we cannot solve this model numerically. 

Instead, we follow Krusell-Smith and approximate the distribution with a finite set of moments, which here we denote $K$ (aggregate capital). 

## 2) Model with Krusell-Smith Approximation

### HH problem
Individual state is $s=(k,e,z,K)$, where $k$ is assets, $e\in\{0,1\}$ is employment, $z\in\{z_b,z_g\}$ is aggregate productivity, and $K$ is aggregate capital.
Given prices and an aggregate forecasting rule, the Bellman equation is
$$
V(k,e,z,K)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',K')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,K))k + w(z,K)e \\
k'\ge k_{min} \\
K' = \mathcal{H}(K,z)
$$


### Firms problem
We assume a representative firm with production function:
$$
Y = zK^\alpha N^{1-\alpha},
$$
and competitive factor prices are
$$
r = \alpha z (K/N)^{\alpha-1}-\delta,\qquad
w = (1-\alpha)z(K/N)^\alpha.
$$

### Equilibrium definition
A recursive equilibrium is a collection
$$
\{V,\,g,\mathcal{H},\,r,\,w,\,\Psi\},
$$

such that:
1. Given $(r(z,K),w(z,K))$ and perceived law of motion $\mathcal{H}$, value function $V$ and policy function $g$ solve the household problem.
2. Given $z$ and $r(z,K), w(z,K)$, firm maximizes profits, in particular:
$$
r(z,K) = \alpha z (K/N)^{\alpha-1}-\delta,\qquad
w(z,K) = (1-\alpha)z(K/N)^\alpha.
$$
3. Given $g$ and shock transitions, we can obtain the actual law of motion for aggregate capital:
    $$
    K' = \mathcal{G}(K,z)
    $$
    The actual law of motion $\mathcal{G}$ is consistent with the perceived law of motion $\mathcal{H}$
4. Market clearing:
$$
K = \int k\,d\Psi(k,e,z,K) \\
N = \int e\,d\Psi(k,e,z,K)
$$

Krusell-Smith approximation specifies a particular functional form for the perceived law of motion $\mathcal{H}$, which is log-linear:
$$
\log K' = a_z + a_z \log K
$$

## 3) General Numerical Algorithm (Inner/Outer Iteration)

We use a nested iteration algorithm.

### Inner loop: solve HH problem
Given coefficients $(a_z,b_z)$, we map $K$ to predicted $K'$ and solve HH problem:
$$
V(k,e,z,K)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\,\mathbb E\left[V(k',e',z',K')\mid e,z\right]\right\} \quad\text{s.t.}\quad \\

c + k' = (1+r(z,K))k + w(z,K)e \\
k'\ge k_{min} \\
log K' = a_z + b_z \log K
$$

At each state $(k,e,z,K)$ we search over $k'\in\mathcal K$ and compute expected continuation value using transition probabilities for $(z,e)\to(z',e')$.
We iterate until value function converged:
$$
\|V_{new}-V\|_\infty < \varepsilon_V.
$$

### Outer loop: update aggregate law of motion
With the policy function $g(k,e,z,K)$ solved from the inner loop, we:
1. Simulate many households and periods.
2. Construct time series data $\{K_t,z_t\}$.
3. Run state-by-state regressions
$$
\log K_{t+1}=a_z+b_z\log K_t+\epsilon_{t+1},\qquad z\in\{b,g\}.
$$
4. Update perceived coefficients with damping
$$
\theta^{new}=\lambda\theta^{old}+(1-\lambda)\hat\theta,\quad \theta=(a_b,b_b,a_g,b_g).
$$
5. Repeat until stopping criterion is met (in this notebook: both $R^2$ exceed a threshold).


### Code Cell 1: Utilities and Grid Mapping
This cell defines numerical helpers used everywhere else.

The household utility is CRRA:
$$u(c)=\begin{cases}\log(c), & \gamma=1\\\frac{c^{1-\gamma}-1}{1-\gamma}, & \gamma\neq 1\end{cases}$$
with a large penalty when $c\le 0$ to enforce feasibility.

It also maps a continuous value to the closest grid point:
$$i(x)=\arg\min_j\,|x-\text{grid}_j|.$$

In [6]:
using Random, Statistics, LinearAlgebra, Printf

util(c, gamma) = c <= 0 ? -1.0e12 : (gamma == 1.0 ? log(c) : (c^(1.0 - gamma) - 1.0) / (1.0 - gamma))

function nearest_index(grid::Vector{Float64}, x::Float64)
    j = searchsortedfirst(grid, x)
    if j <= 1
        return 1
    elseif j > length(grid)
        return length(grid)
    else
        return abs(grid[j] - x) < abs(grid[j - 1] - x) ? j : j - 1
    end
end

nearest_index (generic function with 1 method)

### Code Cell 2: KS Baseline Parameters and Economic Primitives
This cell sets the baseline calibration and core objects.

Model parameters are grouped in one struct:
$$\theta=(\beta,\gamma,\alpha,\delta,\{z\},P_z,P_{ze}).$$

Prices come from a Cobb-Douglas production side:
$$r=\alpha z\left(\frac{K}{N}\right)^{\alpha-1}-\delta,\qquad
w=(1-\alpha)z\left(\frac{K}{N}\right)^{\alpha}.$$

It also builds individual and aggregate capital grids for numerical solution.

In [7]:
Base.@kwdef mutable struct KSParams
    beta::Float64 = 0.99
    gamma::Float64 = 1.0
    alpha::Float64 = 0.36
    delta::Float64 = 0.025
    z_vals::Vector{Float64} = [0.99, 1.01]                          # [bad, good]
    N_vals::Vector{Float64} = [0.2944, 0.3140]                      # aggregate labor from KS baseline
    eff_labor::Float64 = 0.3271                                     # employed labor efficiency
    unemp_transfer::Float64 = 0.07                                  # unemployment transfer
    Pz::Matrix{Float64} = [0.875 0.125; 0.125 0.875]               # aggregate productivity transition
    Pze::Matrix{Float64} = [0.5250 0.3500 0.0312 0.0938;           # (z,e)->(z',e')
                            0.0389 0.8361 0.0021 0.1229;
                            0.0938 0.0312 0.2917 0.5833;
                            0.0091 0.1159 0.0243 0.8507]
    u_rate::Vector{Float64} = [0.10, 0.04]                          # initial employment draw
    k_min::Float64 = 0.0
    k_max::Float64 = 80.0
    nk::Int = 120
    K_min::Float64 = 1.0
    K_max::Float64 = 45.0
    nK::Int = 45
    N_agents::Int = 5000
    T_sim::Int = 1200
    burn_in::Int = 200
end

# State ordering: 1=(bad,unemp), 2=(bad,emp), 3=(good,unemp), 4=(good,emp)
state_index(ie::Int, iz::Int) = (iz - 1) * 2 + ie

function make_grids(par::KSParams)
    kgrid = collect(range(par.k_min, par.k_max, length = par.nk))
    Kgrid = collect(range(par.K_min, par.K_max, length = par.nK))
    return kgrid, Kgrid
end

function prices(par::KSParams, K::Float64, z::Float64, N::Float64)
    Kn = K / max(N, 1.0e-8)
    r = par.alpha * z * Kn^(par.alpha - 1.0) - par.delta
    w = (1.0 - par.alpha) * z * Kn^(par.alpha)
    return r, w
end

prices (generic function with 1 method)

### Code Cell 3: Bellman Operator and Policy Function
This cell implements the inner fixed point $V=T(V)$.

At each grid state $(k,e,z,K)$, it computes
$$
(TV)(k,e,z,K)=\max_{k'\in\mathcal K}\left\{u(c)+\beta\sum_{z',e'}P_{ze}((z,e),(z',e'))V(k',e',z',K')\right\}.
$$
The argmax gives the savings policy $g(k,e,z,K)$.

The cell therefore returns:
1. Value function $V$.
2. Policy indices for $k'$.

Convergence check is the sup norm:
$$
\max_{k,e,z,K}|V_{new}-V|<\text{tol}.
$$

In [8]:
function solve_household(par::KSParams, kgrid, Kgrid, coeff; max_iter = 2000, tol = 1e-5)
    nk, nK = length(kgrid), length(Kgrid)
    V = zeros(nk, nK, 2, 2)
    Vnew = similar(V)
    pol = ones(Int, nk, nK, 2, 2)

    for it in 1:max_iter
        diff = 0.0
        for iz in 1:2
            z = par.z_vals[iz]
            N = par.N_vals[iz]
            for iK in 1:nK
                K = Kgrid[iK]
                r, w = prices(par, K, z, N)
                Knext = exp(coeff[iz, 1] + coeff[iz, 2] * log(K))
                iKnext = nearest_index(Kgrid, Knext)

                for ie in 1:2
                    labor_income = ie == 2 ? w * par.eff_labor : par.unemp_transfer
                    cur_state = state_index(ie, iz)
                    for ik in 1:nk
                        cash = (1.0 + r) * kgrid[ik] + labor_income
                        best_v = -1.0e18
                        best_ikp = 1

                        for ikp in 1:nk
                            kp = kgrid[ikp]
                            c = cash - kp
                            u = util(c, par.gamma)
                            if u <= -1.0e11
                                continue
                            end

                            ev = 0.0
                            for izp in 1:2
                                for iep in 1:2
                                    nxt_state = state_index(iep, izp)
                                    p = par.Pze[cur_state, nxt_state]
                                    ev += p * V[ikp, iKnext, iep, izp]
                                end
                            end

                            val = u + par.beta * ev
                            if val > best_v
                                best_v = val
                                best_ikp = ikp
                            end
                        end

                        Vnew[ik, iK, ie, iz] = best_v
                        pol[ik, iK, ie, iz] = best_ikp
                        diff = max(diff, abs(best_v - V[ik, iK, ie, iz]))
                    end
                end
            end
        end

        V, Vnew = Vnew, V
        if it % 50 == 0
            @printf("  VFI iter %d, diff = %.3e\n", it, diff)
        end
        if diff < tol
            @printf("  VFI converged at iter %d, diff = %.3e\n", it, diff)
            return V, pol
        end
    end

    @warn "VFI reached max_iter without full convergence"
    return V, pol
end

solve_household (generic function with 1 method)

### Code Cell 4: Simulation, Aggregation, and KS Outer Fixed Point
This cell closes the equilibrium approximation step.

Given a policy $g$:
1. Simulate $(z_t,e_{i,t},k_{i,t})$.
2. Aggregate to $K_t=\frac{1}{N}\sum_i k_{i,t}$.
3. Estimate aggregate law coefficients via OLS in each aggregate state.

The estimated law is
$$
\log K_{t+1}=a_z+b_z\log K_t,
$$
and coefficients are damped before the next iteration to improve stability.

This is the key KS consistency step: perceived law $\to$ optimal policy $\to$ simulated law $\to$ updated perceived law.

Current stopping rule in code:
$$
R^2_{bad}>\text{r2\_tol}\quad\text{and}\quad R^2_{good}>\text{r2\_tol}.
$$

In [9]:
function draw_next_z(iz::Int, Pz::Matrix{Float64}, rng)

    udraw = rand(rng)

    return udraw <= Pz[iz, 1] ? 1 : 2

end



function p_emp_next(par::KSParams, ie::Int, iz::Int, izp::Int)

    cur_state = state_index(ie, iz)

    pz = par.Pz[iz, izp]

    if pz <= 1.0e-12

        return ie == 2 ? 1.0 : 0.0

    end

    emp_state = state_index(2, izp)

    p_emp = par.Pze[cur_state, emp_state] / pz

    return clamp(p_emp, 0.0, 1.0)

end



function simulate_economy(par::KSParams, kgrid, Kgrid, pol, coeff; seed = 2026)

    rng = MersenneTwister(seed)

    T, N = par.T_sim, par.N_agents



    k_idx = fill(cld(length(kgrid), 2), N)

    e_state = [rand(rng) < (1.0 - par.u_rate[2]) ? 2 : 1 for _ in 1:N]

    z_path = ones(Int, T + 1)

    z_path[1] = 2



    K_series = zeros(T + 1)

    K_series[1] = mean(kgrid[k_idx])



    for t in 1:T

        iz = z_path[t]

        Kt = mean(kgrid[k_idx])

        K_series[t] = Kt

        iK = nearest_index(Kgrid, Kt)



        izp = draw_next_z(iz, par.Pz, rng)

        z_path[t + 1] = izp



        k_next_idx = similar(k_idx)

        for i in 1:N

            ie = e_state[i]

            ik = k_idx[i]

            ikp = pol[ik, iK, ie, iz]

            k_next_idx[i] = ikp



            p_emp = p_emp_next(par, ie, iz, izp)

            e_state[i] = rand(rng) < p_emp ? 2 : 1

        end



        k_idx = k_next_idx

        K_series[t + 1] = mean(kgrid[k_idx])

    end



    return K_series, z_path

end



function fit_law_of_motion(K_series, z_path; burn_in = 200)

    coeff = zeros(2, 2)

    r2 = zeros(2)



    for z in 1:2

        idx = [t for t in burn_in:(length(K_series) - 1) if z_path[t] == z]

        x = log.(K_series[idx])

        y = log.(K_series[idx .+ 1])



        X = hcat(ones(length(x)), x)

        b = X \ y

        coeff[z, :] .= b



        yhat = X * b

        ssr = sum((y .- yhat) .^ 2)

        sst = sum((y .- mean(y)) .^ 2)

        r2[z] = 1.0 - ssr / max(sst, 1.0e-12)

    end



    return coeff, r2

end



function solve_ks(par::KSParams; max_outer = 20, damping = 0.9, r2_tol = 0.9999, base_seed = 2026, vary_seed = true)
    kgrid, Kgrid = make_grids(par)

    coeff = [0.05 0.95; 0.05 0.95]



    V = zeros(length(kgrid), length(Kgrid), 2, 2)

    pol = ones(Int, length(kgrid), length(Kgrid), 2, 2)

    K_series = zeros(par.T_sim + 1)

    z_path = ones(Int, par.T_sim + 1)

    r2 = zeros(2)



    for out in 1:max_outer

        @printf("\n=== Outer iteration %d ===\n", out)



        V, pol = solve_household(par, kgrid, Kgrid, coeff)

        seed_used = vary_seed ? (base_seed + out) : base_seed
        K_series, z_path = simulate_economy(par, kgrid, Kgrid, pol, coeff; seed = seed_used)
        new_coeff, r2 = fit_law_of_motion(K_series, z_path; burn_in = par.burn_in)



        gap = maximum(abs.(new_coeff .- coeff))

        @printf("  bad state : a = %.4f, b = %.4f, R2 = %.4f\n", new_coeff[1,1], new_coeff[1,2], r2[1])

        @printf("  good state: a = %.4f, b = %.4f, R2 = %.4f\n", new_coeff[2,1], new_coeff[2,2], r2[2])

        @printf("  max coefficient gap = %.4e\n", gap)



        coeff .= damping .* coeff .+ (1.0 - damping) .* new_coeff



        if (r2[1] > r2_tol) && (r2[2] > r2_tol)

            @printf("Converged: outer loop stopped at iter %d (R2 thresholds met).\n", out)

            return (; coeff, r2, V, pol, kgrid, Kgrid, K_series, z_path)

        end

    end



    @warn "Outer loop reached max_outer"

    return (; coeff, r2, V, pol, kgrid, Kgrid, K_series, z_path)

end

solve_ks (generic function with 1 method)

### Code Cell 5: Run the Solver and Report Results
This cell runs the full pipeline:
1. Initialize parameters.
2. Solve the model.
3. Print estimated law-of-motion coefficients.

Reported form is:
$$\log K' = a_z + b_z\log K,$$
with one pair $(a_z,b_z)$ for each aggregate state $z$.

In [ ]:
par = KSParams()

res = solve_ks(par; max_outer = 20, damping = 0.9, r2_tol = 0.9999, vary_seed = true)


println("\nFinal perceived law of motion (rows: bad, good):")

println("[a_z  b_z] =")

display(res.coeff)

println("R2 by state = " , res.r2)



println("\nBad-state law:  log(K') = $(round(res.coeff[1,1], digits=4)) + $(round(res.coeff[1,2], digits=4))*log(K)")

println("Good-state law: log(K') = $(round(res.coeff[2,1], digits=4)) + $(round(res.coeff[2,2], digits=4))*log(K)")


=== Outer iteration 1 ===
  VFI iter 50, diff = 1.482e+00


## 3) Extensions
- Calibrate employment transitions exactly as in the original paper.
- Use interpolation (instead of nearest grid point) for \(K'\).
- Add acceleration methods (Howard, parallel loops, EGM-style approximations).